# Kaggriculture | Adaptive Farm Intelligence — Route Portfolio Exact

A compact, reproducible Kaggriculture submission package.

## Method

An exact, self-contained reproduction of V21-R1 Public-State Route Portfolio. The published route-selection logic and action programme are unchanged.

The notebook writes the required `submission.tar.gz`.

In [ ]:
# Self-contained competition package.
import base64
import hashlib
import io
import tarfile
from pathlib import Path

ARCHIVE_B64 = (
    '...'
)
EXPECTED_SHA256 = "d65410a4fc8a31226055e43085131468ac8af5721aabdf3593cb8eade59cabe3"
payload = base64.b64decode(ARCHIVE_B64)
assert hashlib.sha256(payload).hexdigest() == EXPECTED_SHA256
with tarfile.open(fileobj=io.BytesIO(payload), mode="r:gz") as archive:
    assert archive.getnames() == ["main.py"]
    source = archive.extractfile("main.py").read()
    compile(source, "main.py", "exec")
Path("submission.tar.gz").write_bytes(payload)
print("submission.tar.gz ready")
print("sha256:", EXPECTED_SHA256)
print("agent source:", len(source), "bytes")
